In [1]:
!ollama list

NAME             	ID          	SIZE  	MODIFIED     
llama3.2:latest  	a80c4f17acd5	2.0 GB	19 hours ago	
bge-m3:latest    	790764642607	1.2 GB	19 hours ago	
all-minilm:latest	1b226e2802db	45 MB 	19 hours ago	
llama3.2:1b      	baf6a787fdff	1.3 GB	19 hours ago	
smollm:135m      	b0b2a4617438	91 MB 	19 hours ago	


In [2]:
import os
import json
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings
from llama_index.core.llama_dataset.generator import RagDatasetGenerator
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.prompts import ChatPromptTemplate, ChatMessage, MessageRole, PromptTemplate
from typing import List, Dict, Any
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import TreeSummarize
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine, CitationQueryEngine
from llama_index.core.types import BaseModel
from pydantic import BaseModel
from llama_index.core import Document
from llama_index.vector_stores.chroma import ChromaVectorStore

In [3]:
with open('configs/gcp.env', 'r') as json_file:
    config = json.load(json_file)

In [4]:
print(config)

{'collection': 'thoi-su', 'data_dir': '../data', 'doc_folder': 'results', 'index_folder': 'urls', 'storage_folder': 'db', 'embedding_model': 'bge-m3:latest', 'llm_model': 'llama3.2:latest', 'timeout': 60000, 'top_docs': 20, 'top_buffer': 3, 'max_questions': 2}


In [5]:
doc_path = config['data_dir'] + "/" + config['doc_folder'] + "/" + config["collection"]
print(doc_path)

../data/results/thoi-su


In [6]:
# Step 2: Load text files and URLs
doc_files = os.listdir(doc_path)
print(f"Number of doc_files: {len(doc_files)}")

Number of doc_files: 619


In [7]:
index_path = config['data_dir'] + "/" + config['index_folder'] + "/" + config["collection"] + ".txt"
print(index_path)

../data/urls/thoi-su.txt


In [8]:
with open(index_path, 'r') as f:
    url_list = f.read().splitlines()

In [9]:
# Check for missing files between txt files and URL list
overlaps = set([i for i in range(1, len(doc_files)+1) if f"url_{i:03}.txt" in doc_files])
missing = sorted(set([i for i in range(1, len(doc_files)+1)]) - overlaps)
print(f"Missing: {missing}")

Missing: [13, 23, 76, 151, 364, 370, 384, 403]


In [10]:
timeout = int(config['timeout'])
print(f"Timeout: {timeout}")

Timeout: 60000


In [11]:
# Step 3: Initialize index and storage
Settings.embed_model = OllamaEmbedding(config["embedding_model"], timeout=timeout)
Settings.llm = Ollama(model=config["llm_model"], timeout=timeout)

In [12]:
# Load documents from the directory
documents = SimpleDirectoryReader(doc_path).load_data()

In [13]:
# Create a persistent Chroma client and collection in the database
storage_path = config['data_dir'] + "/" + config['storage_folder'] + "/" + config["collection"]
print(storage_path)
db = chromadb.PersistentClient(storage_path)

../data/db/thoi-su


In [14]:
collection_name = config["collection"].replace("-", "_")
print(collection_name)

thoi_su


In [15]:
try:
    # Attempt to delete the existing collection
    db.delete_collection(name=collection_name)
    print(f"Collection '{collection_name}' has been deleted.")
except ValueError as e:
    # Handle case where collection does not exist
    if "does not exist" in str(e):
        print(f"No existing collection named '{collection_name}' to delete.")
    else:
        raise e  # Re-raise unexpected errors

Collection 'thoi_su' has been deleted.


In [16]:
collection = db.create_collection(collection_name, metadata={"hnsw:space": config["embedding_space"]})

In [17]:
# Assign Chroma as vector store and create an index from documents
vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [18]:
doc_index = VectorStoreIndex.from_documents(documents, storage_context=storage_context, show_progress=True)

Parsing nodes:   0%|          | 0/619 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1320 [00:00<?, ?it/s]

In [19]:
doc_retriever = doc_index.as_retriever(
    similarity_top_k=config["top_docs"],
    verbose=True
)

In [20]:

response_prompt_template = PromptTemplate(
    "Câu hỏi gốc: {query_str}\n"
    "Ngữ cảnh đi kèm: {orig_query}\n"
    "Tài liệu tham khảo:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Trả lời câu hỏi gốc dưới đây một cách chi tiết, biết ngữ cảnh đi kèm của câu hỏi được đính vào phía dưới đây cũng như các tài liệu tham khảo để trả lời câu hỏi được cung cấp sẵn. Câu trả lời phải thật rõ ràng, cụ thể và đầy đủ.\n"
    "Câu trả lời bằng tiếng Việt: "
)

response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
    text_qa_template=response_prompt_template,
    structured_answer_filtering=True,
    verbose=True
)


In [21]:

query_prompt_template = PromptTemplate(
    "Câu truy vấn của người dùng: {query_str}\n"
    "Cuộc trò chuyện quá khứ như sau:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Dựa trên dữ liệu cuộc trò chuyện, hãy viết lại câu truy vấn của người dùng dưới dạng một câu hỏi duy nhất có ngữ cảnh sao cho không cần biết cuộc trò chuyện trước đó vẫn đọc hiểu câu truy vấn. Nếu cuộc trò chuyện quá khứ không đủ để làm rõ câu truy vấn, giữ lại y nguyên câu truy vấn. Câu truy vấn phải được viết lại thành một câu hỏi duy nhất mà không kèm câu trả lời.\n"
    "Câu hỏi bằng tiếng Việt: "
)

query_synthesizer = get_response_synthesizer(
    response_mode="refine",
    text_qa_template=query_prompt_template,
    # structured_answer_filtering=True,
)


In [22]:
question_prompt_template = PromptTemplate("""\
Với ngữ liệu tham khảo dưới đây:
---------------------
{context_str}
---------------------
Dựa trên những ngữ liệu tham khảo này, hãy đưa ra chỉ những câu hỏi mà không kèm câu trả lời cho câu truy vấn dưới đây.
Mỗi câu hỏi phải được viết dưới dạng câu hỏi hoàn chỉnh và cụ thể nhất có thể. Câu truy vấn:
{query_str}
Chỉ đưa ra các câu hỏi bằng tiếng Việt:
"""
)

In [23]:
# Step 10: Implement chatbot interaction loop with all features
def chatbot():
    print("Chatbot is ready! Type 'exit' to end the chat.")
    past_query = ""
    conversation_iter = 0
    mem_index = VectorStoreIndex([], show_progress=False)
    
    while True:
        mem_retriever = mem_index.as_retriever(
            similarity_top_k=config["top_buffer"],
            verbose=True,
        )
        
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            print("Chatbot: Goodbye!")
            break
        
        mem_nodes = mem_retriever.retrieve(user_input)
        query_input = query_synthesizer.synthesize(user_input, mem_nodes)
        
        if len(query_input.source_nodes) == 0:
            query_inp = user_input
        else:
            query_inp = query_input.response
            
        print("Query: ", query_inp)
        doc_nodes = doc_retriever.retrieve(query_inp)
        
        chatbot_response = response_synthesizer.synthesize(query_inp, doc_nodes, original_query=user_input)
        response = chatbot_response.response
        print("Chatbot: ", response)
        
        query = f"Người dùng: {user_input}\n" + f"Cụ thể hơn: {user_input}\n" + f"Chatbot: {response}\n"
        concat_query = "Dữ liệu cuộc trò chuyện quá khứ\n." + past_query + "\nLượt kế:\n" + query
        
        # print("-"*20 + "References" + "-"*20)
        # print(concat_query)            
        # print("-"*20 + "End References" + "-"*20)
        
        doc = Document(text=concat_query, doc_id=f"turn_{conversation_iter}")
        mem_index.insert(doc)
        
        adjusted_ques_num = 2 * config["max_questions"] + 1
        dg = RagDatasetGenerator.from_documents([doc], text_question_template=question_prompt_template,
                                                num_questions_per_chunk=adjusted_ques_num)
        questions = dg.generate_questions_from_nodes().to_pandas()
        followups = questions.loc[questions['query'].str.endswith('?'), "query"].values
        
        print("Follow-ups", followups)
        
        past_query = query
        conversation_iter += 1

In [24]:
import nest_asyncio
nest_asyncio.apply()

In [25]:
chatbot()

Chatbot is ready! Type 'exit' to end the chat.
Query:  từ thiện và bão lũ
5 text chunks after repacking
1 text chunks after repacking
Chatbot:  Việc quyên góp tài trợ cho người dân bị ảnh hưởng bởi thiên tai, đặc biệt là trong trường hợp của bão lũ.
Follow-ups ['Người dùng muốn hỗ trợ ai trong trường hợp bão lũ?'
 'Chatbot đang yêu cầu quyên góp tài trợ gì cho người dân bị ảnh hưởng bởi thiên tai?'
 'Trong trường hợp bão lũ xảy ra, người dùng có thể làm gì để giúp đỡ?'
 'Những trường hợp thiên tai nào mà chatbot thường xuyên nhắc đến trong hoạt động từ thiện?']
Query:  "Kẻ thiện từ có thể quyên góp tài trợ cho người dân bị ảnh hưởng bởi bão lũ không?"
5 text chunks after repacking
1 text chunks after repacking
Chatbot:  Có.
Follow-ups ['Người dùng "từ thiện" đang nói chuyện với ai?'
 'Việc quyên góp tài trợ cho người dân bị ảnh hưởng bởi thiên tai là gì?'
 'Người dùng "đánh bóng" đang làm gì?'
 'Chatbot đã trả lời câu hỏi nào của người dùng "từ thiện"?']
Query:  Bão Yagi là một cơn bão